# Phase 7 — PySpark Performance Engineering Experiments Notebook

This is the **STARTER notebook** for Phase 7.

Run it **top-to-bottom**. The setup, deterministic retail datasets, helper
functions, temporary Parquet environment, experiment order, and diagnostic
prompts are preserved from the SOLUTION notebook.

Worked experiment code and answer-revealing explanations have been removed so
you can complete the performance diagnosis yourself.

Use the required workflow throughout:

```text
baseline
    ↓
predict bottleneck
    ↓
inspect evidence
    ↓
execute / measure
    ↓
change ONE thing
    ↓
inspect again
    ↓
execute / measure again
    ↓
compare
    ↓
reconcile correctness
    ↓
explain WHY
```

Core rule:

> **Fix query and data design before tuning configuration.**

**Scope:** Phase 7 practice only. This notebook does not perform the formal
mastery gate, update `ROADMAP.md`, mark Phase 7 complete, or enter Phase 8
Spark UI diagnostics.


<a id="toc"></a>
## Table of Contents

- [Setup and Practice Data](#setup-and-practice-data)
- [Experiment Protocol](#experiment-protocol)
- [Experiment 1 — Diagnose Unnecessary I/O](#experiment-1)
- [Experiment 2 — File Sizing: Fix the Writer](#experiment-2)
- [Experiment 3 — Broadcast vs. Shuffle Join](#experiment-3)
- [Experiment 4 — Remove an Unnecessary Shuffle](#experiment-4)
- [Experiment 5 — Diagnose Skew and Apply Salting](#experiment-5)
- [Experiment 6 — Justified vs. Unnecessary Caching](#experiment-6)
- [Experiment 7 — AQE Runtime Improvements](#experiment-7)
- [Applied Phase 7 Project](#applied-project)

---

<a id="setup-and-practice-data"></a>
# Setup and Practice Data

Main grains:

```text
fact_sales_df
= one row per sale_id

dim_store_df
= one row per store_id

dim_product_df
= one row per product_id

skewed_sales_df
= one row per sale_id
```


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


spark = (
    SparkSession.builder
    .appName('phase_07_performance_experiments')
    .master('local[4]')
    # Keep ordinary experiments static. AQE is enabled only in Experiment 7.
    .config('spark.sql.shuffle.partitions', '12')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate()
)


In [ ]:
# Grain: one row per sale_id.
fact_sales_df = (
    spark.range(
        start=0,
        end=120000,
        step=1,
        numPartitions=8,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.date_add(
            F.lit('2026-01-01').cast('date'),
            (F.col('id') % 180).cast('int'),
        ).alias('order_date'),
        F.concat(
            F.lit('S'),
            F.lpad(
                ((F.col('id') % 40) + F.lit(1)).cast('string'),
                3,
                '0',
            ),
        ).alias('store_id'),
        F.concat(
            F.lit('P'),
            F.lpad(
                ((F.col('id') % 500) + F.lit(1)).cast('string'),
                4,
                '0',
            ),
        ).alias('product_id'),
        F.concat(
            F.lit('C'),
            F.lpad(
                ((F.col('id') % 8000) + F.lit(1)).cast('string'),
                5,
                '0',
            ),
        ).alias('customer_id'),
        F.when(
            (F.col('id') % 10) < 8,
            F.lit('COMPLETED'),
        )
        .when(
            (F.col('id') % 10) == 8,
            F.lit('CANCELLED'),
        )
        .otherwise(F.lit('RETURNED'))
        .alias('order_status'),
        ((F.col('id') % 5) + F.lit(1)).cast('int').alias('quantity'),
        (
            F.lit(5.00)
            + ((F.col('id') % 75) * F.lit(0.75))
        )
        .cast(DecimalType(12, 2))
        .alias('unit_price'),
        F.concat(
            F.lit('PROMO_'),
            (F.col('id') % 25).cast('string'),
        ).alias('promotion_code'),
        F.concat(
            F.lit('CHANNEL_'),
            (F.col('id') % 4).cast('string'),
        ).alias('sales_channel'),
        F.concat(
            F.lit('DEVICE_'),
            (F.col('id') % 6).cast('string'),
        ).alias('device_type'),
    )
    .withColumn('year', F.year('order_date'))
    .withColumn('month', F.month('order_date'))
    .withColumn(
        'gross_sales',
        (F.col('quantity') * F.col('unit_price')).cast(DecimalType(16, 2)),
    )
)


# Grain: one row per store_id.
dim_store_df = (
    spark.range(
        start=1,
        end=41,
        step=1,
        numPartitions=2,
    )
    .select(
        F.concat(
            F.lit('S'),
            F.lpad(F.col('id').cast('string'), 3, '0'),
        ).alias('store_id'),
        F.concat(
            F.lit('Store '),
            F.col('id').cast('string'),
        ).alias('store_name'),
        F.when(F.col('id') <= 20, F.lit('ON'))
        .otherwise(F.lit('BC'))
        .alias('province'),
        F.when((F.col('id') % 2) == 0, F.lit('URBAN'))
        .otherwise(F.lit('SUBURBAN'))
        .alias('store_format'),
    )
)


# Grain: one row per product_id.
dim_product_df = (
    spark.range(
        start=1,
        end=501,
        step=1,
        numPartitions=4,
    )
    .select(
        F.concat(
            F.lit('P'),
            F.lpad(F.col('id').cast('string'), 4, '0'),
        ).alias('product_id'),
        F.concat(
            F.lit('Product '),
            F.col('id').cast('string'),
        ).alias('product_name'),
        F.concat(
            F.lit('CATEGORY_'),
            (F.col('id') % 12).cast('string'),
        ).alias('category'),
        F.concat(
            F.lit('BRAND_'),
            (F.col('id') % 30).cast('string'),
        ).alias('brand'),
    )
)


# Grain: one row per sale_id.
# HOT_STORE deliberately owns 70% of the rows.
skewed_sales_df = (
    spark.range(
        start=0,
        end=100000,
        step=1,
        numPartitions=8,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.when(
            F.col('id') < 70000,
            F.lit('HOT_STORE'),
        )
        .otherwise(
            F.concat(
                F.lit('COLD_'),
                F.lpad(
                    ((F.col('id') % 30) + F.lit(1)).cast('string'),
                    2,
                    '0',
                ),
            )
        )
        .alias('store_id'),
        ((F.col('id') % 5) + F.lit(1)).cast('int').alias('quantity'),
        (
            F.lit(10.00)
            + ((F.col('id') % 25) * F.lit(0.50))
        )
        .cast(DecimalType(12, 2))
        .alias('unit_price'),
    )
    .withColumn(
        'gross_sales',
        (F.col('quantity') * F.col('unit_price')).cast(DecimalType(16, 2)),
    )
)


In [ ]:
def show_partition_summary(df, label):
    '''Print execution-partition count and non-empty row distribution.'''

    rows_by_partition = (
        df
        .select(F.spark_partition_id().alias('partition_id'))
        .groupBy('partition_id')
        .agg(F.count('*').alias('row_count'))
        .orderBy('partition_id')
        .collect()
    )

    print(f'\n{label}')
    print('execution partitions:', df.rdd.getNumPartitions())

    for row in rows_by_partition:
        print(
            f'partition {row.partition_id}: '
            f'{row.row_count} rows'
        )


def time_count(df, label):
    '''Materialize the same count workload and report elapsed local time.'''

    started_at = perf_counter()
    row_count = df.count()
    elapsed_seconds = perf_counter() - started_at

    print(f'{label}: {row_count} rows in {elapsed_seconds:.3f} s')
    return row_count, elapsed_seconds


def parquet_data_files(path):
    '''Return only physical Parquet data files below a local path.'''

    return sorted(Path(path).rglob('*.parquet'))


def show_file_summary(path, label):
    '''Print compact physical Parquet file-size evidence.'''

    files = parquet_data_files(path)
    sizes = [file_path.stat().st_size for file_path in files]

    print(f'\n{label}')
    print('parquet data files:', len(files))

    if sizes:
        print('minimum bytes:', min(sizes))
        print('maximum bytes:', max(sizes))
        print('average bytes:', round(sum(sizes) / len(sizes), 2))
        print('total bytes:', sum(sizes))


def scalar_sum(df, column_name):
    '''Return one deterministic numeric reconciliation value.'''

    return (
        df
        .agg(F.sum(column_name).alias('value'))
        .first()
        .value
    )


def assert_same_rows(df_a, df_b, columns):
    '''Assert exact multiset equality for deterministic teaching outputs.'''

    left = df_a.select(*columns)
    right = df_b.select(*columns)

    assert left.exceptAll(right).count() == 0
    assert right.exceptAll(left).count() == 0


In [ ]:
# Keep this object alive for the entire notebook session.
temp_directory = TemporaryDirectory(prefix='pyspark_phase_07_')
temp = Path(temp_directory.name)

partitioned_sales_path = temp / 'fact_sales_partitioned'

(
    fact_sales_df
    .write
    .mode('overwrite')
    .partitionBy('year', 'month')
    .parquet(str(partitioned_sales_path))
)

print('temporary root:', temp)
print('fact rows:', fact_sales_df.count())
print('fact execution partitions:', fact_sales_df.rdd.getNumPartitions())


[Back to Table of Contents](#toc)

---

<a id="experiment-protocol"></a>
# Experiment Protocol

For every experiment, keep the business result fixed and change exactly one
performance variable.

```text
Business requirement
Input grain(s)
Output grain
Correctness invariants
Suspected bottleneck
Evidence
Materializing action
```

Then compare physical behavior, runtime support, and correctness.


[Back to Table of Contents](#toc)

---

<a id="experiment-1"></a>
# Experiment 1 — Diagnose Unnecessary I/O

**Requirement:** completed June 2026 sales by store.

**Output grain:** one row per `store_id`.

The partitioned Parquet dataset is physically organized by `year` and `month`.
The baseline filters June using only `order_date`, which is logically correct but
does not directly constrain the stored partition columns.

### Prediction

- Required business columns: `store_id`, `quantity`, `gross_sales`.
- `order_status` is needed for row filtering.
- The baseline date predicate may appear in `PushedFilters`.
- The baseline should lack direct `year` / `month` `PartitionFilters`.
- Equivalent `year` / `month` predicates should enable partition pruning.


In [ ]:
# TODO: Write your solution here.


### Your baseline diagnosis

Before changing anything, write:

```text
Likely bottleneck:
Plan/file/partition evidence:
Why this is the most relevant first problem:
What should remain unchanged for a fair comparison:
```


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


### Your explanation

After both runs, explain:

```text
Bottleneck:
Evidence:
One change:
Before → after physical behavior:
Runtime evidence:
Correctness evidence:
Why performance changed:
```

Be precise about the difference between column pruning, predicate pushdown, and
partition pruning.


[Back to Table of Contents](#toc)

---

<a id="experiment-2"></a>
# Experiment 2 — File Sizing: Fix the Writer

The same logical fact data is written twice.

```text
baseline: 96 output execution partitions
one change: 8 output execution partitions
```

No schema, codec, storage partitioning, writer file-cap, or read setting changes.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


### Your explanation

Compare the two physical layouts and answer:

1. How did writer execution partition count affect data-file count?
2. How did file-size distribution change?
3. How did the next read's execution partition count change?
4. Did elapsed time support the expected direction?
5. Why is `fewer files = better` an unsafe universal rule?
6. Why should a recurring small-file problem usually be fixed at the writer?


[Back to Table of Contents](#toc)

---

<a id="experiment-3"></a>
# Experiment 3 — Broadcast vs. Shuffle Join

**Requirement:** enrich every sale with store attributes.

```text
fact_sales_df = many rows per store_id
dim_store_df = one row per store_id
output grain = one row per sale_id
```

Validate the dimension grain before judging join performance.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


### Your explanation

Compare the plans and answer:

1. Which join operator appears in the baseline?
2. Which `Exchange` / `Sort` requirements support that strategy?
3. Which join operator appears after the one change?
4. What happened to large-side data movement?
5. Why was broadcasting logically safe here?
6. What uniqueness/grain check protects correctness?


[Back to Table of Contents](#toc)

---

<a id="experiment-4"></a>
# Experiment 4 — Remove an Unnecessary Shuffle

**Requirement:** completed sales totals by `product_id`.

Baseline smell:

```text
repartition(12, 'store_id')
→ groupBy('product_id')
```

The first distribution does not satisfy the second wide requirement.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


### Your explanation

Explain:

```text
Which Exchange was unnecessary?
Which Exchange remains necessary?
Why did the removed distribution not help the downstream operator?
Why is the goal not to eliminate every shuffle?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-5"></a>
# Experiment 5 — Diagnose Skew and Apply Targeted Salting

`HOT_STORE` deliberately owns 70% of the data.

The first question is not merely “how many partitions?” but “how are rows
distributed across the key space?”


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


### Your diagnosis

Use the key-frequency table and execution-partition distribution to answer:

1. Is this mainly a partition-count problem or a distribution problem?
2. What share of rows belongs to the hottest key?
3. Why would increasing the number of hash partitions alone not split that key?
4. What evidence would justify targeted salting?


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


### Your explanation

After implementing the salted version, explain:

1. Why must the salt vary *within* the hot key?
2. Why would `hash(store_id)` alone fail?
3. What is the intermediate salted grain?
4. Why is a second aggregation required?
5. How do you prove that salting preserved the original business result?
6. What simpler fixes should be considered before manual salting?


[Back to Table of Contents](#toc)

---

<a id="experiment-6"></a>
# Experiment 6 — Justified vs. Unnecessary Caching

Cache only when the same expensive lineage is reused enough to justify its
storage cost.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


### Your caching diagnosis

Compare uncached, cache-materializing, and cache-reusing behavior.

Explain:

```text
Why is this intermediate a plausible cache candidate?
What work does materialization add?
What later work can reuse the cache?
Why would caching a one-use DataFrame be wasteful?
Why is caching the reduced intermediate preferable to caching a wider raw input?
When should unpersist() be called?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-7"></a>
# Experiment 7 — AQE Runtime Improvements

AQE is evaluated after the static query/data design is reasonable.

Inspect what Spark actually does. Runtime join conversion and skew splitting may
or may not trigger on a local teaching workload.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


### Part A — your AQE partition-coalescing analysis

Record:

```text
planned/static shuffle partition count:
final adaptive execution partition count:
adaptive plan evidence:
runtime comparison:
```

Explain whether AQE coalesced post-shuffle partitions in *your* run.


In [ ]:
# TODO: Write your solution here.


### Part B — your AQE runtime-join analysis

Inspect the final adaptive plan.

Record:

```text
initial join operator:
final join operator:
did runtime broadcast conversion occur?:
evidence:
```

If Spark does not change the join, say so rather than inferring a conversion.


In [ ]:
# TODO: Write your solution here.


### Part C — your AQE skew analysis

Use the already-proven hot-key distribution plus the final adaptive plan.

Record:

```text
data skew present?:
AQE skew handling enabled?:
did the final plan show skew splitting/handling?:
evidence:
```

Do not claim AQE handled skew unless the runtime plan shows it.


In [ ]:
# TODO: Write your solution here.


[Back to Table of Contents](#toc)

---

<a id="applied-project"></a>
# Applied Phase 7 Project — Diagnose and Optimize One Pipeline

**Business requirement:** June 2026 completed sales by `province` and `category`
with `units` and `gross_sales`.

Expected output grain:

```text
one row per (province, category)
```

The baseline is logically correct but intentionally inefficient. The worked
solution chooses one first optimization and leaves the other candidates intact.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


## Your applied diagnosis

Before optimizing, identify and rank at least three candidate bottlenecks.

Use:

```text
Candidate 1:
Evidence:
Estimated importance:

Candidate 2:
Evidence:
Estimated importance:

Candidate 3:
Evidence:
Estimated importance:
```

Then choose exactly **one** first optimization and justify why it should have the
highest expected impact while preserving the required output grain.


In [ ]:
# TODO: Write your solution here.


In [ ]:
# TODO: Write your solution here.


## Your final diagnosis

Complete:

```text
Bottleneck:
Evidence:
First optimization:
Before → after physical behavior:
Runtime result:
Correctness validation:
Why performance changed:
Next bottleneck to investigate:
```

Do not apply the next optimization in the same experiment.


[Back to Table of Contents](#toc)

---

# Phase 7 Practice Checklist — Mastery Gate Not Yet Performed

Before considering the practice notebook finished, confirm that you have
independently implemented and explained:

```text
[ ] pruning diagnosis
[ ] file-sizing comparison
[ ] broadcast vs. shuffle join
[ ] unnecessary shuffle removal
[ ] skew / hot-key diagnosis
[ ] salting with correctness reconciliation
[ ] justified caching / persistence
[ ] AQE runtime inspection
[ ] integrated one-change-at-a-time diagnosis
```

For any memory/resource symptom, first identify the query/data cause before
proposing executor/driver tuning.

The formal Phase 7 mastery gate remains separate.
